In [7]:
from semanticscholar import SemanticScholar
from crossref.restful import Works
from itertools import product
import pandas as pd
import json
import dblp
import requests

In [8]:
crossref_work = Works()
sch = SemanticScholar()

In [9]:
# search params
primary_keywords = [
    "Generative AI",
    "LLM",
    "LM",
    "GenAI",
    "large language model",
    "language model",
    "codex",
    "gpt",
    "gpt-3",
    "gpt-4",
]
secondary_keywords = [
    "overreliance",
    "misinformation",
    "accessbility",
    "privacy",
    "enviromental",
    "explainability",
    "trustworthy",
    "responsible",
]
others = ["mitigation", "ethics", "societal", "social", "ethical"]


year = "2019"
years = ["2019", "2020", "2021", "2022", "2023"]
dblp_formatted_years = " " + "|".join(f"{year}" for year in years)

all_combinations = list(product(primary_keywords, secondary_keywords, others))

In [10]:
# Prepare DataFrame to store the results
columns = [
    'PaperTitle',
    'DOI',    
    'SearchString',
    'SearchedFrom',
]
search_results_df = pd.DataFrame(columns=columns)

In [11]:
# SemanticScholar fields
fields = [
    "title",
    "externalIds",
    "paperId",
    "url"
]

In [12]:
try:
    for search_string in all_combinations:
        search_string_formatted = ' '.join(search_string)
        print(f"-------------------------------searching for {search_string_formatted}------------------------------")
        
        # Semantic Scholar
         # send http request to api
        api_url = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"
        headers = {"Content-Type": "application/json"}
        params = {"query": search_string_formatted, "year": year, "fields": ",".join(fields)}
        response = requests.get(api_url, headers=headers, params=params)
        response_json = response.json()
        sch_results = response_json["data"]
        print(f"Semantic scholar total: {response_json['total']}")
        # Add results to DataFrame
        sch_count = 0
        if sch_results != 0:
            for result in sch_results:
                sch_count += 1
                if result['externalIds'].get('DOI') is None:
                    if result['externalIds'].get('ArXiv') is None:
                        doi = f"no-doi, sch url: {result.get('url')}"
                    else:
                        doi =  f"10.48550/arXiv.{result['externalIds'].get('ArXiv')}"
                else:
                    doi = result['externalIds'].get('DOI')
                new_paper = {
                    'PaperTitle': result['title'],
                    'DOI': doi,
                    'SearchString': search_string_formatted,
                    'SearchedFrom': 'Semantic Scholar'
                }
                print(f"{sch_count}. sch process paper: ", new_paper)
                search_results_df = pd.concat([search_results_df, pd.DataFrame([new_paper])], ignore_index=True) 
        
        # # # Crossref
        # # TODO: too many result
        # cr_search_results = crossref_work.query(search_string_formatted).filter(from_online_pub_date=year)
        # print(f"Crossref total: {cr_search_results.count()}")
        # crossref_count = 0
        # for cr_result in cr_search_results:
        #     crossref_count += 1
        #     new_paper = {
        #         'PaperTitle': cr_result.get('title'),
        #         'DOI': cr_result.get('DOI'),
        #         'SearchString': search_string_formatted,
        #         'SearchedFrom': 'Crossref'
        #     }
        #     print(f"{crossref_count}. Crossref process paper: ", new_paper)
        #     search_results_df = pd.concat([search_results_df, pd.DataFrame([new_paper])], ignore_index=True) 

        # search in dblp
        dblp_search_results = dblp.search(search_string_formatted + dblp_formatted_years)
        if dblp_search_results is None:
            print(f"DBLP total: 0")
        else:
            print(f"DBLP total: {len(dblp_search_results)}")
            dblp_count = 0
            for key, result in dblp_search_results.items():
                dblp_count += 1
                new_paper = {
                    'PaperTitle': result.get('title'),
                    'DOI': result.get('doi'),
                    'SearchString': search_string_formatted,
                    'SearchedFrom': 'DBLP'
                }
                print(f"{dblp_count}. DBLP process paper: ", new_paper)
                search_results_df = pd.concat([search_results_df, pd.DataFrame([new_paper])], ignore_index=True)
        print(f"^^^^^^^^^^^^^^^^^^^^^^^^^^^searching end for {search_string_formatted}^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n\n")
    search_results_df.to_csv('data/search-results.csv')
except Exception as e:
    search_results_df.to_csv('data/search-results.csv')
    print(f"An error occurred: {e.with_traceback()}")


-------------------------------searching for Generative AI overreliance mitigation------------------------------
Semantic scholar total: 0
DBLP total: 0
^^^^^^^^^^^^^^^^^^^^^^^^^^^searching end for Generative AI overreliance mitigation^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


-------------------------------searching for Generative AI overreliance ethics------------------------------
Semantic scholar total: 0
DBLP total: 0
^^^^^^^^^^^^^^^^^^^^^^^^^^^searching end for Generative AI overreliance ethics^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


-------------------------------searching for Generative AI overreliance societal------------------------------
Semantic scholar total: 0
DBLP total: 0
^^^^^^^^^^^^^^^^^^^^^^^^^^^searching end for Generative AI overreliance societal^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


-------------------------------searching for Generative AI overreliance social------------------------------
Semantic scholar total: 0
DBLP total: 0
^^^^^^^^^^^^^^^^^^^^^^^^^^^searching end for Generati

KeyboardInterrupt: 

# Testing stuff

In [ ]:
search_string = 'Ormco Unveils SymetriTM Clear ceramic twin bracket system'
sch_test = sch.search_paper(search_string)

In [ ]:
search_string = 'Generative AI AND Social Impact AND education'
test =  cr_search_results = crossref_work.query(search_string).count()

In [ ]:
search_string = 'generative ai' + " 2020|2021"
dblp_test = dblp.search(search_string)
dblp_test.items()

In [ ]:
api_url = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"
headers = {"Content-Type": "application/json"}
params = {"query": "Generative AI + misinformation + social", "year": year, "fields": ",".join(fields)}
response = requests.get(api_url, headers=headers, params=params)
response_json = response.json()